Training LSTM with and without weather


In [17]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict, Optional
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# Configuration
FILE_PATH = '/RawData_MeasuredHeadDemand.csv'
TIME_COL = 'Time Point'
TARGET_COL = 'Measured Heat Demand[W]'

In [18]:
class DataCleaner:
    @staticmethod
    def clean_and_validate(df: pd.DataFrame, time_col: str, target_col: str) -> pd.DataFrame:
        df = df.copy()
        if time_col in df.columns:
            df[time_col] = pd.to_datetime(df[time_col])
            df = df.set_index(time_col)
        df = df.loc[~df.index.duplicated(keep='first')]
        df = df.asfreq('h')
        return df

class TimeSeriesImputer:
    def __init__(self, strategy: str = 'historical_avg'):
        valid_strategies = ['historical_avg', 'linear_interpolation']
        if strategy not in valid_strategies:
            raise ValueError(f"Strategy must be one of {valid_strategies}")
        self.strategy = strategy
        self.hourly_means_: Optional[Dict[int, float]] = None

    def fit(self, X: pd.DataFrame, target_col: str):
        if self.strategy == 'historical_avg':
            self.hourly_means_ = X.groupby(X.index.hour)[target_col].mean().to_dict()
        return self

    def transform(self, X: pd.DataFrame, target_col: str) -> pd.DataFrame:
        X_out = X.copy()
        if self.strategy == 'linear_interpolation':
            X_out[target_col] = X_out[target_col].interpolate(method='linear')
        elif self.strategy == 'historical_avg':
            if self.hourly_means_ is None:
                raise RuntimeError("Imputer must be fitted first.")
            mapped_means_index = X_out.index.hour.map(self.hourly_means_)
            mapped_means_series = pd.Series(mapped_means_index, index=X_out.index)
            X_out[target_col] = X_out[target_col].fillna(mapped_means_series)

        X_out[target_col] = X_out[target_col].ffill().bfill()
        return X_out

class FeatureEngineer:
    @staticmethod
    def create_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['hour'] = df.index.hour
        df['day_of_week'] = df.index.dayofweek
        df['month'] = df.index.month
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        return df

class DataSplitter:
    @staticmethod
    def time_based_split(df: pd.DataFrame, test_months: int = 3) -> Tuple[pd.DataFrame, pd.DataFrame]:
        end_date = df.index.max()
        split_date = end_date - pd.DateOffset(months=test_months)
        train_df = df[df.index < split_date].copy()
        test_df = df[df.index >= split_date].copy()
        return train_df, test_df

class BaselineForecasters:
    def __init__(self, y_train: pd.Series):
        self.train_mean = y_train.mean()

    def _safe_shift(self, series: pd.Series, periods: int) -> pd.Series:
        shifted = series.shift(periods)
        shifted = shifted.ffill()
        shifted = shifted.fillna(self.train_mean)
        return shifted

    def predict_daily_naive(self, full_series: pd.Series) -> pd.Series:
        r"""Forecast: \hat{y}_t = y_{t-24}"""
        return self._safe_shift(full_series, periods=24)

    def predict_weekly_naive(self, full_series: pd.Series) -> pd.Series:
        r"""Forecast: \hat{y}_t = y_{t-168}"""
        return self._safe_shift(full_series, periods=168)

    def predict_combined_seasonal(self, full_series: pd.Series) -> pd.Series:
        r"""Forecast: \hat{y}_t = \frac{1}{2}(y_{t-24} + y_{t-168})"""
        daily_forecast = self.predict_daily_naive(full_series)
        weekly_forecast = self.predict_weekly_naive(full_series)
        return (daily_forecast + weekly_forecast) / 2

In [19]:
def calculate_metrics(actual, pred) -> dict:
    """Calculates standard forecasting metrics."""
    return {
        'RMSE': np.sqrt(mean_squared_error(actual, pred)),
        'MAE': mean_absolute_error(actual, pred),
        'MAPE (%)': mean_absolute_percentage_error(actual, pred) * 100
    }

def run_pipeline_and_evaluate(strategy: str, df: pd.DataFrame, time_col: str, target_col: str) -> list:
    """Runs the full pipeline for a given imputation strategy and returns metrics."""
    # 1. Clean Data
    df_cleaned = DataCleaner.clean_and_validate(df, time_col=time_col, target_col=target_col)

    # 2. Split Data
    train_df, test_df = DataSplitter.time_based_split(df_cleaned, test_months=3)

    # 3. Impute Data
    imputer = TimeSeriesImputer(strategy=strategy)
    train_df = imputer.fit(train_df, target_col=target_col).transform(train_df, target_col=target_col)
    test_df = imputer.transform(test_df, target_col=target_col)

    # 4. Generate Features (needed for ML later, good to include in pipeline)
    train_df = FeatureEngineer.create_temporal_features(train_df)
    test_df = FeatureEngineer.create_temporal_features(test_df)

    # 5. Initialize Baselines & Forecast
    baselines = BaselineForecasters(y_train=train_df[target_col])
    full_target_series = pd.concat([train_df[target_col], test_df[target_col]])

    daily_preds = baselines.predict_daily_naive(full_target_series).loc[test_df.index]
    weekly_preds = baselines.predict_weekly_naive(full_target_series).loc[test_df.index]
    combined_preds = baselines.predict_combined_seasonal(full_target_series).loc[test_df.index]

    # 6. Collect Metrics
    test_actuals = test_df[target_col]
    return [
        {'Strategy': strategy, 'Model': 'Seasonal Naive (Daily)', **calculate_metrics(test_actuals, daily_preds)},
        {'Strategy': strategy, 'Model': 'Seasonal Naive (Weekly)', **calculate_metrics(test_actuals, weekly_preds)},
        {'Strategy': strategy, 'Model': 'Combined Seasonal', **calculate_metrics(test_actuals, combined_preds)}
    ]

In [20]:
# Load the raw data once
print("Loading data...")
raw_df = pd.read_csv(FILE_PATH)
raw_df[TIME_COL] = pd.to_datetime(raw_df[TIME_COL], utc=True)

# Run the evaluation loop
strategies = ['historical_avg', 'linear_interpolation']
all_results = []

for strat in strategies:
    strat_results = run_pipeline_and_evaluate(strat, raw_df, TIME_COL, TARGET_COL)
    all_results.extend(strat_results)

# Display a beautiful summary table
results_df = pd.DataFrame(all_results)
results_df['RMSE'] = results_df['RMSE'].apply(lambda x: f"{x:,.0f}")
results_df['MAE'] = results_df['MAE'].apply(lambda x: f"{x:,.0f}")
results_df['MAPE (%)'] = results_df['MAPE (%)'].apply(lambda x: f"{x:.2f}%")

print("\n" + "="*80)
print("BASELINE FORECASTING PERFORMANCE COMPARISON")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

Loading data...

BASELINE FORECASTING PERFORMANCE COMPARISON
            Strategy                   Model      RMSE       MAE MAPE (%)
      historical_avg  Seasonal Naive (Daily) 2,151,582 1,277,222   13.96%
      historical_avg Seasonal Naive (Weekly) 3,327,804 2,375,743   28.60%
      historical_avg       Combined Seasonal 2,232,318 1,560,708   18.33%
linear_interpolation  Seasonal Naive (Daily) 2,127,707 1,264,493   13.76%
linear_interpolation Seasonal Naive (Weekly) 3,324,672 2,373,108   28.51%
linear_interpolation       Combined Seasonal 2,221,147 1,553,996   18.21%


In [21]:
# Prepare global training and testing sets for SARIMA and LSTM
# Using linear_interpolation as it showed slightly better baseline performance

# 1. Clean
df_cleaned = DataCleaner.clean_and_validate(raw_df, time_col=TIME_COL, target_col=TARGET_COL)

# 2. Split
train_df, test_df = DataSplitter.time_based_split(df_cleaned, test_months=3)

# 3. Impute
imputer = TimeSeriesImputer(strategy='historical_avg')
train_df = imputer.fit(train_df, target_col=TARGET_COL).transform(train_df, target_col=TARGET_COL)
test_df = imputer.transform(test_df, target_col=TARGET_COL)

# 4. Feature Engineering
train_df = FeatureEngineer.create_temporal_features(train_df)
test_df = FeatureEngineer.create_temporal_features(test_df)

# 5. Define y_train/y_test for SARIMA (specifically requested by those cells)
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]

print(f"Data prepared successfully.")
print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")

Data prepared successfully.
Training set shape: (15383, 8)
Testing set shape: (2161, 8)


LSTM

In [22]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer

class RevIN(Layer):
    """
    Reversible Instance Normalization (RevIN) Layer for Time-Series Forecasting.
    Normalizes input sequences per instance to mitigate distribution shift,
    and denormalizes the output predictions using the stored instance statistics.
    """
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True, **kwargs):
        super(RevIN, self).__init__(**kwargs)
        self.num_features = num_features
        self.eps = eps
        self.affine = affine

    def build(self, input_shape):
        if self.affine:
            self.gamma = self.add_weight(
                name="gamma",
                shape=(self.num_features,),
                initializer="ones",
                trainable=True
            )
            self.beta = self.add_weight(
                name="beta",
                shape=(self.num_features,),
                initializer="zeros",
                trainable=True
            )
        super(RevIN, self).build(input_shape)

    def call(self, inputs, mode='norm'):
        """
        Args:
            inputs: Tensor of shape (batch_size, sequence_length, num_features) for 'norm'
                    or (batch_size, horizon, num_features) / (batch_size, num_features) for 'denorm'.
            mode: 'norm' for forward instance normalization, 'denorm' for reverting the scaling.
        """
        if mode == 'norm':
            # Calculate and store instance-level statistics across the sequence length (axis 1)
            self.mean = tf.reduce_mean(inputs, axis=1, keepdims=True)
            self.var = tf.math.reduce_variance(inputs, axis=1, keepdims=True)
            self.stdev = tf.sqrt(self.var + self.eps)

            # Apply instance normalization
            x = (inputs - self.mean) / self.stdev

            # Apply learnable affine transformation
            if self.affine:
                x = x * self.gamma + self.beta
            return x

        elif mode == 'denorm':
            x = inputs
            # Revert learnable affine transformation
            if self.affine:
                x = (x - self.beta) / self.gamma

            # Handle dimension broadcasting if the model outputs a 2D tensor (e.g., single step forecast)
            if len(x.shape) == 2:
                x = tf.expand_dims(x, axis=1)
                x = x * self.stdev + self.mean
                x = tf.squeeze(x, axis=1)
            else:
                x = x * self.stdev + self.mean

            return x

    def get_config(self):
        config = super(RevIN, self).get_config()
        config.update({
            "num_features": self.num_features,
            "eps": self.eps,
            "affine": self.affine
        })
        return config


import numpy as np

def create_sequences_ci(data: np.ndarray, target_col_idx: int, lookback: int = 168):
    """
    Creates aligned sequence windows for a Channel Independent (CI) forecasting model.
    Separates the historical target sequence from the future temporal covariates.

    Args:
        data (np.ndarray): The 2D array of features (shape: [num_samples, num_features]).
        target_col_idx (int): The index of the column to be used as the prediction target.
        lookback (int): The number of historical time steps to include in the input sequence.

    Returns:
        X_target (np.ndarray): Historical target sequences of shape (samples, lookback, 1).
        X_covariates (np.ndarray): Future temporal covariates of shape (samples, num_covariates).
        y (np.ndarray): Target outputs of shape (samples,).
    """
    horizon = 1
    if len(data) <= lookback + horizon:
        raise ValueError("Data length must be greater than lookback window.")

    num_features = data.shape[1]
    # Identify indices for the temporal covariates (everything except the target)
    covariate_cols = [i for i in range(num_features) if i != target_col_idx]

    X_target, X_covariates, y = [], [], []

    # Slide the window across the dataset
    for i in range(len(data) - lookback):
        # 1. Extract the historical sequence of ONLY the target
        # Using [:, target_col_idx : target_col_idx + 1] keeps the shape as (lookback, 1)
        window_x_target = data[i : i + lookback, target_col_idx : target_col_idx + 1]
        X_target.append(window_x_target)

        # 2. Extract the temporal covariates for the FUTURE hour we are predicting
        window_x_covariates = data[i + lookback, covariate_cols]
        X_covariates.append(window_x_covariates)

        # 3. Extract the true future target value
        window_y = data[i + lookback, target_col_idx]
        y.append(window_y)

    return np.array(X_target), np.array(X_covariates), np.array(y)

In [23]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Add, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.losses import Huber
from tensorflow.keras.optimizers import Adam

def build_ci_lstm(target_input_shape: tuple, covariate_input_shape: tuple) -> Model:
    """
    Builds a Channel Independent Dual-Input Stacked Bidirectional LSTM.
    Separates historical target sequences (normalized via RevIN) from future covariates.
    """
    # 1. Dual Input Layers
    target_input = Input(shape=target_input_shape, name="target_sequence_input")
    covariate_input = Input(shape=covariate_input_shape, name="future_covariate_input")

    # 2. Reversible Instance Normalization (Applied ONLY to the target history)
    revin_layer = RevIN(num_features=target_input_shape[-1], affine=True, name="revin_layer")
    x_norm = revin_layer(target_input, mode='norm')

    # 3. Stacked Bidirectional LSTMs processing the normalized history
    lstm_1 = Bidirectional(LSTM(64, return_sequences=True, activation='tanh'), name="bilstm_1")(x_norm)
    lstm_2 = Bidirectional(LSTM(64, return_sequences=True, activation='tanh'), name="bilstm_2")(lstm_1)

    # Residual Connection
    res_connection = Add(name="residual_add")([lstm_1, lstm_2])

    # Compress sequence into a single vector representation
    lstm_3 = Bidirectional(LSTM(32, return_sequences=False, activation='tanh'), name="bilstm_3")(res_connection)

    # 4. Merge Extracted History Features with Future Covariates
    merged = Concatenate(name="feature_merge")([lstm_3, covariate_input])

    # 5. Dense Projection (Output size must match the isolated target feature count for denormalization)
    dense_out = Dense(target_input_shape[-1], name="dense_projection")(merged)

    # 6. Denormalization (Reverts back to the original Heat Demand scale)
    final_output = revin_layer(dense_out, mode='denorm')

    # 7. Model Compilation
    model = Model(inputs=[target_input, covariate_input], outputs=final_output, name="CI_ResBiLSTM")

    optimizer = Adam(learning_rate=1e-3)
    model.compile(
        optimizer=optimizer,
        loss=Huber(delta=1.0),
        metrics=['mse', 'mae']
    )

    return model

In [24]:
# 1. Prepare data for the Advanced LSTM
# Combine train and test for feature extraction, keeping them separate for scaling logic in RevIN
lookback = 168
target_idx = 0 # 'Measured Heat Demand[W]' is the first column[cite: 1]

# Convert DataFrames to numpy arrays (Ensuring this step is still here)
train_data = train_df.values
test_data = pd.concat([train_df.iloc[-lookback:], test_df]).values

# 2. Create separated CI sequences for training and testing
X_train_target, X_train_covariates, y_train = create_sequences_ci(
    data=train_data,
    target_col_idx=target_idx,
    lookback=lookback
)

X_test_target, X_test_covariates, y_test = create_sequences_ci(
    data=test_data,
    target_col_idx=target_idx,
    lookback=lookback
)

print("Channel Independent Sequence Creation Complete.")
print(f"X_train_target shape:     {X_train_target.shape}")      # Expected: (15215, 168, 1)
print(f"X_train_covariates shape: {X_train_covariates.shape}")  # Expected: (15215, 7)
print(f"y_train shape:            {y_train.shape}")             # Expected: (15215,)

# 3. Define the new input shapes for the CI model
target_input_shape = (X_train_target.shape[1], X_train_target.shape[2]) # (168, 1)
covariate_input_shape = (X_train_covariates.shape[1],)                  # (7,)

print(f"Target Input Shape: {target_input_shape}")
print(f"Covariate Input Shape: {covariate_input_shape}")

# Initialize the dual-input model
model = build_ci_lstm(
    target_input_shape=target_input_shape,
    covariate_input_shape=covariate_input_shape
)

model.summary()

Channel Independent Sequence Creation Complete.
X_train_target shape:     (15215, 168, 1)
X_train_covariates shape: (15215, 7)
y_train shape:            (15215,)
Target Input Shape: (168, 1)
Covariate Input Shape: (7,)


Model: "CI_ResBiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ target_sequence_in… │ (None, 168, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ revin_layer (RevIN) │ (None, 1)         │          2 │ target_sequence_… │
│                     │                   │            │ dense_projection… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 168, 128)  │     33,792 │ revin_layer[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 168, 128)  │     98,816 │ bilstm_1[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_add (Add)  │ (None, 168, 128)  │          0 │ bilstm_1[0][0],   │
│                     │                   │            │ bilstm_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_3            │ (None, 64)        │     41,216 │ residual_add[0][… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ future_covariate_i… │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_merge       │ (None, 71)        │          0 │ bilstm_3[0][0],   │
│ (Concatenate)       │                   │            │ future_covariate… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_projection    │ (None, 1)         │         72 │ feature_merge[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 173,898 (679.29 KB)

 Trainable params: 173,898 (679.29 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray, model_name: str = "Advanced LSTM"):
    """
    Calculates and prints standard forecasting metrics.
    Ensures predictions are flattened to 1D arrays for accurate metric calculation.
    """
    # Flatten arrays to prevent broadcasting errors
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print("\n" + "="*60)
    print(f"{model_name.upper()} PERFORMANCE (TEST SET)")
    print("="*60)
    print(f"RMSE: {rmse:,.0f}")
    print(f"MAE:  {mae:,.0f}")
    print(f"MAPE: {mape:.2f}%")
    print("="*60)

    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

def evaluate_forecast(y_true: np.ndarray, y_pred: np.ndarray, model_name: str = "Advanced LSTM"):
    """Calculates and prints standard forecasting metrics."""
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print("\n" + "="*60)
    print(f"{model_name.upper()} PERFORMANCE (TEST SET)")
    print("="*60)
    print(f"RMSE: {rmse:,.0f}")
    print(f"MAE:  {mae:,.0f}")
    print(f"MAPE: {mape:.2f}%")
    print("="*60)

    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

print("--- Step 4: Configuring Advanced Callbacks ---")
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)

print("\n--- Step 5: Training the CI Model ---")
# Notice we are now passing a LIST of inputs for X
history = model.fit(
    x=[X_train_target, X_train_covariates],
    y=y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

print("\n--- Step 6: Inference & Evaluation ---")
# Inference also requires the list of inputs
advanced_lstm_preds = model.predict([X_test_target, X_test_covariates])

metrics = evaluate_forecast(
    y_true=y_test,
    y_pred=advanced_lstm_preds,
    model_name="Channel Independent ResBiLSTM"
)

--- Step 4: Configuring Advanced Callbacks ---

--- Step 5: Training the CI Model ---
Epoch 1/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - loss: 857772.1250 - mae: 857772.5625 - mse: 1924608688128.0000 - val_loss: 960799.4375 - val_mae: 960800.0625 - val_mse: 2691015639040.0000 - learning_rate: 0.0010
Epoch 2/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - loss: 310827.1562 - mae: 310827.6562 - mse: 298923524096.0000 - val_loss: 481969.5312 - val_mae: 481970.0938 - val_mse: 1204250083328.0000 - learning_rate: 0.0010
Epoch 3/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - loss: 222274.8750 - mae: 222275.3906 - mse: 198005899264.0000 - val_loss: 460862.4375 - val_mae: 460862.9062 - val_mse: 1089908965376.0000 - learning_rate: 0.0010
Epoch 4/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - loss: 210554.2812 - mae: 210554.8906 - mse: 192714817536.0000 - val_loss: 440020.1875 - val_mae: 440020.6562 - val_mse: 1018913030144.0000 - learning_rate: 0.0010
Epoch 5/100
214/214 ━━━━━━━━━━━━━━━━

integrating weather data and training the model on that

In [30]:
import pandas as pd

def merge_datasets(demand_path: str, weather_path: str, time_col: str) -> pd.DataFrame:
    """
    Loads, standardizes, and merges demand and weather datasets.
    Forces a strict hourly frequency, generating NaNs for missing readings.
    """
    print("Loading datasets...")
    # Load raw CSVs
    df_demand = pd.read_csv(demand_path)

    # FIX 1: We must actually load the weather data into the df_weather variable!
    df_weather = pd.read_csv(weather_path)

    # Standardize Datetime to UTC to avoid daylight saving time ambiguities
    df_demand[time_col] = pd.to_datetime(df_demand[time_col], utc=True)
    df_weather[time_col] = pd.to_datetime(df_weather[time_col], utc=True)

    # Set indices for alignment
    df_demand.set_index(time_col, inplace=True)
    df_weather.set_index(time_col, inplace=True)

    print("Merging on temporal index...")
    # Outer join to ensure no timestamps from either dataset are dropped
    df_merged = df_demand.join(df_weather, how='outer')

    # Drop any exact duplicate index rows that might have snuck in from raw data
    df_merged = df_merged.loc[~df_merged.index.duplicated(keep='first')]

    # Force strict hourly frequency (injects NaNs for entirely missing hours)
    df_merged = df_merged.asfreq('h')

    print(f"Merge complete. New shape: {df_merged.shape}")
    return df_merged

# ==========================================
# Pipeline Execution Script
# ==========================================

# 1. Configuration
# FIX 2: Removed the leading '/' so Python looks in your current local Colab folder
DEMAND_FILE = '/RawData_MeasuredHeadDemand.csv'
WEATHER_FILE = '/weather_data.csv'#'/RawData_ExternalTemperature.csv'
TIME_COL = 'Time Point'
TARGET_COL = 'Measured Heat Demand[W]'
WEATHER_COL = 'Temperature'
COLUMNS_TO_IMPUTE = [TARGET_COL, WEATHER_COL]

# 2. Merge and Resample
df_master = merge_datasets(
    demand_path=DEMAND_FILE,
    weather_path=WEATHER_FILE,
    time_col=TIME_COL
)

# 3. Split Data BEFORE Imputation (Crucial for preventing data leakage)
# Assuming DataSplitter is defined in your environment from previous cells
train_df, test_df = DataSplitter.time_based_split(df_master, test_months=3)

# 4. Independent Imputation
# We instantiate a fresh imputer for each column so the historical averages
# (e.g., average temp at 2 PM vs. average demand at 2 PM) do not overwrite each other.
imputers = {}

for col in COLUMNS_TO_IMPUTE:
    print(f"Applying historical average imputation to: {col}")
    imputers[col] = TimeSeriesImputer(strategy='historical_avg')

    # Fit strictly on the training data to learn historical averages
    train_df = imputers[col].fit(train_df, target_col=col).transform(train_df, target_col=col)

    # Transform test data using the averages learned ONLY from the training set
    test_df = imputers[col].transform(test_df, target_col=col)

print("Pipeline execution complete. Train and Test sets are merged, aligned, and imputed safely.")

Loading datasets...
Merging on temporal index...
Merge complete. New shape: (17544, 5)
Applying historical average imputation to: Measured Heat Demand[W]
Applying historical average imputation to: Temperature
Pipeline execution complete. Train and Test sets are merged, aligned, and imputed safely.


In [31]:
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# Pipeline Execution Script (Updated)
# ==========================================

# 1. Configuration & Merge (From previous step)
df_master = merge_datasets(DEMAND_FILE, WEATHER_FILE, TIME_COL)

# 2. Split Data BEFORE Imputation and Scaling (Crucial for preventing data leakage)
train_df, test_df = DataSplitter.time_based_split(df_master, test_months=3)

# 3. Independent Imputation (From previous step)
imputers = {}
for col in COLUMNS_TO_IMPUTE:
    imputers[col] = TimeSeriesImputer(strategy='historical_avg')
    train_df = imputers[col].fit(train_df, target_col=col).transform(train_df, target_col=col)
    test_df = imputers[col].transform(test_df, target_col=col)

# 4. Exogenous Variable Scaling
print("Applying MinMaxScaler strictly to Exogenous Weather data...")
temp_scaler = MinMaxScaler()

# Fit the scaler STRICTLY on the training set.
# Note: sklearn expects a 2D array, so we use double brackets [[WEATHER_COL]]
temp_scaler.fit(train_df[[WEATHER_COL]])

# Transform both training and test sets using the bounds learned from the past
train_df[WEATHER_COL] = temp_scaler.transform(train_df[[WEATHER_COL]])
test_df[WEATHER_COL] = temp_scaler.transform(test_df[[WEATHER_COL]])

print(f"Temperature scaled. Train range: [{train_df[WEATHER_COL].min():.2f}, {train_df[WEATHER_COL].max():.2f}]")

# 5. Temporal Feature Engineering
# (These bypass the scaler entirely, remaining bounded to their natural cyclical scale)
print("Generating cyclical temporal features...")
train_df = FeatureEngineer.create_temporal_features(train_df)
test_df = FeatureEngineer.create_temporal_features(test_df)

print("Pipeline data preparation complete. Ready for Channel Independent sequence generation.")

Loading datasets...
Merging on temporal index...
Merge complete. New shape: (17544, 5)
Applying MinMaxScaler strictly to Exogenous Weather data...
Temperature scaled. Train range: [0.00, 1.00]
Generating cyclical temporal features...
Pipeline data preparation complete. Ready for Channel Independent sequence generation.


In [32]:
import numpy as np

def create_sequences_ci(data: np.ndarray, target_col_idx: int, lookback: int = 168):
    """
    Creates aligned sequence windows for a Channel Independent (CI) architecture.
    Separates the historical target sequence from the future covariates (Temporal + Weather).

    Args:
        data (np.ndarray): The 2D array of features (shape: [num_samples, num_features]).
        target_col_idx (int): The index of the column to be used as the prediction target.
        lookback (int): The number of historical time steps to include in the input sequence.

    Returns:
        X_target (np.ndarray): Historical target sequences of shape (samples, lookback, 1).
        X_covariates (np.ndarray): Future temporal & exogenous covariates of shape (samples, num_covariates).
        y (np.ndarray): Target outputs of shape (samples,).
    """
    horizon = 1
    if len(data) <= lookback + horizon:
        raise ValueError("Data length must be greater than lookback window.")

    num_features = data.shape[1]

    # Identify indices for the covariates (EVERYTHING except the target)
    # This now dynamically includes both Temporal Encodings AND Temperature
    covariate_cols = [i for i in range(num_features) if i != target_col_idx]

    X_target, X_covariates, y = [], [], []

    # Slide the window across the dataset
    for i in range(len(data) - lookback):
        # 1. History: Extract past 168 hours of ONLY the target
        window_x_target = data[i : i + lookback, target_col_idx : target_col_idx + 1]
        X_target.append(window_x_target)

        # 2. Future: Extract covariates (Time + Temp) for the EXACT hour we are predicting
        window_x_covariates = data[i + lookback, covariate_cols]
        X_covariates.append(window_x_covariates)

        # 3. Ground Truth: Extract the actual future target value
        window_y = data[i + lookback, target_col_idx]
        y.append(window_y)

    return np.array(X_target), np.array(X_covariates), np.array(y)

# ==========================================
# Sequence Generation & Verification
# ==========================================

# Convert the updated DataFrames to numpy arrays
train_data = train_df.values
test_data = pd.concat([train_df.iloc[-lookback:], test_df]).values

target_idx = 0 # Assuming 'Measured Heat Demand[W]' remains the first column

# Generate the new Dual-Input sequences
X_train_target, X_train_covariates, y_train = create_sequences_ci(train_data, target_idx, lookback)
X_test_target, X_test_covariates, y_test = create_sequences_ci(test_data, target_idx, lookback)

# Define the input shapes for the Keras Model
target_input_shape = (X_train_target.shape[1], X_train_target.shape[2])
covariate_input_shape = (X_train_covariates.shape[1],)

print("Channel Independent Sequence Creation Complete (with Weather Data).")
print(f"X_train_target shape:     {X_train_target.shape}")      # Expected: (samples, 168, 1)
print(f"X_train_covariates shape: {X_train_covariates.shape}")  # Expected: (samples, 8) -> 7 temporal + 1 temp
print(f"y_train shape:            {y_train.shape}")             # Expected: (samples,)
print("-" * 60)
print(f"New Target Input Shape for Model:    {target_input_shape}")
print(f"New Covariate Input Shape for Model: {covariate_input_shape}")

Channel Independent Sequence Creation Complete (with Weather Data).
X_train_target shape:     (15215, 168, 1)
X_train_covariates shape: (15215, 11)
y_train shape:            (15215,)
------------------------------------------------------------
New Target Input Shape for Model:    (168, 1)
New Covariate Input Shape for Model: (11,)


In [33]:
print("--- Step 7: Initializing the Upgraded CI Model ---")
# 1. Initialize the dual-input model using the shapes we just verified
model = build_ci_lstm(
    target_input_shape=target_input_shape,
    covariate_input_shape=covariate_input_shape
)

# Display the architecture to confirm the (None, 8) covariate input layer
model.summary()

print("\n--- Step 8: Configuring Advanced Callbacks ---")
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("\n--- Step 9: Training the Upgraded Model ---")
# 2. Train the model by passing the separated inputs as a list
history = model.fit(
    x=[X_train_target, X_train_covariates],
    y=y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

print("\n--- Step 10: Inference & Evaluation ---")
# 3. Generate predictions using the test set (also passed as a list)
advanced_lstm_preds = model.predict([X_test_target, X_test_covariates])

# 4. Evaluate using your existing evaluate_forecast function
metrics = evaluate_forecast(
    y_true=y_test,
    y_pred=advanced_lstm_preds,
    model_name="Upgraded CI-ResBiLSTM (with Weather)"
)

--- Step 7: Initializing the Upgraded CI Model ---


Model: "CI_ResBiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ target_sequence_in… │ (None, 168, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ revin_layer (RevIN) │ (None, 1)         │          2 │ target_sequence_… │
│                     │                   │            │ dense_projection… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 168, 128)  │     33,792 │ revin_layer[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 168, 128)  │     98,816 │ bilstm_1[0][0]    │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_add (Add)  │ (None, 168, 128)  │          0 │ bilstm_1[0][0],   │
│                     │                   │            │ bilstm_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_3            │ (None, 64)        │     41,216 │ residual_add[0][… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ future_covariate_i… │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_merge       │ (None, 75)        │          0 │ bilstm_3[0][0],   │
│ (Concatenate)       │                   │            │ future_covariate… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_projection    │ (None, 1)         │         76 │ feature_merge[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 173,902 (679.30 KB)

 Trainable params: 173,902 (679.30 KB)

 Non-trainable params: 0 (0.00 B)


--- Step 8: Configuring Advanced Callbacks ---

--- Step 9: Training the Upgraded Model ---
Epoch 1/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step - loss: 7485250.5000 - mae: 7485250.5000 - mse: 329620855455744.0000 - val_loss: 870373.1875 - val_mae: 870373.7500 - val_mse: 2592937345024.0000 - learning_rate: 0.0010
Epoch 2/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - loss: 349232.4062 - mae: 349232.9375 - mse: 338853724160.0000 - val_loss: 695245.1875 - val_mae: 695245.6875 - val_mse: 1727414927360.0000 - learning_rate: 0.0010
Epoch 3/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - loss: 261220.7656 - mae: 261221.2969 - mse: 229873008640.0000 - val_loss: 480838.3125 - val_mae: 480838.9062 - val_mse: 1100506857472.0000 - learning_rate: 0.0010
Epoch 4/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - loss: 227754.0312 - mae: 227754.5625 - mse: 200782528512.0000 - val_loss: 448034.5625 - val_mae: 448035.1250 - val_mse: 1045665021952.0000 - learning_rate: 0.0010
Epoch 5/100
214/214 ━━━━━